In [ ]:
# 쓰레기 분류 퀴즈 앱 - 연못 정화 게임

import torch
import cv2 as cv
from ultralytics import YOLO
import numpy as np
from pathlib import Path
import time
import random
from PIL import ImageFont, ImageDraw, Image

class WasteQuizGame:
    """쓰레기 분류 퀴즈 게임"""
    
    def __init__(self):
        # 화면 크기 설정
        self.screen_width = 700
        self.screen_height = 600
        
        # 한글 폰트 경로 (시스템에 따라 자동 선택)
        self.font_path = self.get_korean_font()
        
        # 퀴즈 문제 데이터베이스
        self.quiz_questions = [
            {
                "question": "플라스틱을 올바르게 분리배출하면 태양광 패널의 어떤 부품으로 활용될 수 있을까요?",
                "options": ["전면 유리", "백시트 소재", "알루미늄 프레임", "전선"],
                "answer": 1,
                "explanation": "플라스틱은 재생하여 태양광 패널 배면의 백시트 소재로 활용됩니다."
            },
            {
                "question": "플라스틱 1kg을 열분해하면 약 얼마나 많은 합성원유를 생산할 수 있을까요?",
                "options": ["0.3L", "0.5L", "0.8L", "1.2L"],
                "answer": 2,
                "explanation": "플라스틱 1kg을 열분해하면 약 0.8L의 합성원유를 생산할 수 있습니다."
            },
            {
                "question": "유리병 1톤을 재활용하면 신규 제조 대비 몇 %의 에너지를 절약할 수 있을까요?",
                "options": ["15%", "32%", "50%", "74%"],
                "answer": 1,
                "explanation": "유리병 1톤 재생 시 신규 제조 대비 32%(약 1,200kWh)의 에너지를 절약합니다."
            },
            {
                "question": "재생 유리는 태양광 산업에서 어디에 활용될까요?",
                "options": ["태양광 패널 전면 유리", "배터리 케이스", "전선 피복", "지지대"],
                "answer": 0,
                "explanation": "재생 유리는 태양광 패널 전면 유리로 재탄생하여 청정에너지를 생산합니다."
            },
            {
                "question": "알루미늄 캔을 재활용하면 신규 제조 대비 몇 %의 에너지를 절약할 수 있을까요?",
                "options": ["50%", "74%", "85%", "95%"],
                "answer": 3,
                "explanation": "알루미늄 재활용은 신규 제조 대비 무려 95%의 에너지를 절약합니다!"
            },
            {
                "question": "재생 철강과 구리는 풍력발전 시설의 어떤 부분에 사용될까요?",
                "options": ["풍력 날개", "제어 시스템", "기초 콘크리트", "터빈 타워와 발전기"],
                "answer": 3,
                "explanation": "재생 철강은 터빈 타워에, 재생 구리는 발전기 부품에 활용됩니다."
            },
            {
                "question": "일반쓰레기 1톤을 소각하면 약 얼마의 전력을 생산할 수 있을까요?",
                "options": ["100~200kWh", "500~700kWh", "1,000~1,200kWh", "2,000~2,500kWh"],
                "answer": 1,
                "explanation": "일반쓰레기 1톤 소각 시 500~700kWh의 전력을 생산하여 난방과 발전에 활용합니다."
            },
            {
                "question": "종이 1톤으로 바이오에너지를 만들면 약 얼마의 열에너지를 생산할까요?",
                "options": ["1,000kWh", "2,000kWh", "3,500kWh", "5,000kWh"],
                "answer": 2,
                "explanation": "종이 1톤으로 약 3,500kWh의 열에너지를 생산할 수 있습니다(가정 3개월 전력량)."
            },
            {
                "question": "비닐 1kg을 열분해하면 약 얼마의 합성연료를 생산할 수 있을까요?",
                "options": ["0.3L", "0.5L", "0.7L", "1.0L"],
                "answer": 2,
                "explanation": "비닐 1kg을 열분해하면 약 0.7L의 합성연료를 생산할 수 있습니다."
            },
            {
                "question": "4인 가족이 올바른 분리배출을 실천하면 연간 약 몇 톤의 CO₂를 감축할 수 있을까요?",
                "options": ["0.8톤", "1.5톤", "2.8톤", "4.2톤"],
                "answer": 2,
                "explanation": "4인 가족 기준 연간 약 2.8톤의 CO₂를 감축할 수 있습니다(소나무 424그루 심는 효과)."
            },
            {
                "question": "종이팩(우유팩)을 일반 폐지와 따로 분리배출해야 하는 이유는?",
                "options": ["냄새가 나서", "부피가 커서", "색깔이 달라서", "고급 펄프를 사용해서"],
                "answer": 3,
                "explanation": "종이팩은 고급 펄프를 사용하여 화장지나 미술용지로 재제조할 수 있습니다."
            },
            {
                "question": "페트병을 찌그러트려 배출하면 어떤 효과가 있을까요?",
                "options": ["운반 효율 증가로 CO₂ 감축", "재활용이 쉬워짐", "분류가 빨라짐", "세척이 편해짐"],
                "answer": 0,
                "explanation": "부피를 줄이면 운반 효율이 높아져 운송 에너지와 CO₂를 절약할 수 있습니다."
            },
            {
                "question": "재생 종이 1톤을 생산하면 몇 그루의 나무를 보호할 수 있을까요?",
                "options": ["5그루", "10그루", "17그루", "25그루"],
                "answer": 2,
                "explanation": "재생 종이 1톤 생산 시 17그루의 나무를 보호하고 4,100kWh의 전력을 절약합니다."
            },
            {
                "question": "전국민이 올바른 분리배출을 실천하면 원자력발전소 몇 기의 발전량과 같은 효과를 낼까요?",
                "options": ["1기", "3기", "5기", "10기"],
                "answer": 1,
                "explanation": "전국민 올바른 분리배출 실천 시 원자력발전소 3기의 연간 발전량과 같은 효과를 냅니다."
            },
            {
                "question": "일반쓰레기를 소각하면 매립에 비해 온실가스를 약 몇 % 감축할 수 있을까요?",
                "options": ["20%", "40%", "60%", "80%"],
                "answer": 2,
                "explanation": "소각 시 열에너지 회수로 매립 대비 온실가스를 약 60% 감축할 수 있습니다."
            }
        ]
        
        self.score = 0
        self.total_questions = 5
        self.current_question = 0
        self.selected_questions = []
        self.pond_pollution = 100  # 연못 오염도 (0 = 깨끗, 100 = 오염)
    
    def get_korean_font(self):
        """시스템에서 한글 폰트 찾기"""
        import platform
        import os
        
        font_paths = []
        
        if platform.system() == 'Windows':
            font_paths = [
                'C:/Windows/Fonts/malgun.ttf',   
                'C:/Windows/Fonts/gulim.ttc',    
            ]
        elif platform.system() == 'Darwin':  
            font_paths = [
                '/System/Library/Fonts/AppleSDGothicNeo.ttc',
                '/Library/Fonts/AppleGothic.ttf',
            ]
        else:  # Linux
            font_paths = [
                '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
                '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
            ]
        
        for font_path in font_paths:
            if os.path.exists(font_path):
                return font_path
        
        # 폰트를 찾지 못한 경우
        print("⚠️ 한글 폰트를 찾을 수 없습니다. 기본 폰트를 사용합니다.")
        return None
    
    def put_korean_text(self, img, text, position, font_size=20, color=(0, 0, 0)):
        """한글 텍스트 그리기"""
        if self.font_path is None:
            # 폰트가 없으면 영어만 표시
            cv.putText(img, text, position, cv.FONT_HERSHEY_SIMPLEX, 
                      font_size/30, color, 2)
            return img
        
        # PIL로 변환
        img_pil = Image.fromarray(img)
        draw = ImageDraw.Draw(img_pil)
        font = ImageFont.truetype(self.font_path, font_size)
        draw.text(position, text, font=font, fill=color)
        
        # OpenCV로 다시 변환
        return np.array(img_pil)
        
    def select_random_questions(self):
        """랜덤으로 5개 문제 선택"""
        self.selected_questions = random.sample(self.quiz_questions, self.total_questions)
        self.score = 0
        self.current_question = 0
        self.pond_pollution = 100
    
    def draw_pond(self, width=500, height=250):
        """연못 그리기 (오염도에 따라 색상 변화)"""
        pond = np.zeros((height, width, 3), dtype=np.uint8)
        
        # 배경 (하늘) - 오염도에 따라 변화
        # 오염됨 (100) -> 회색 하늘: (150, 150, 150)
        # 깨끗함 (0) -> 맑은 하늘: (255, 200, 150)
        pollution_ratio = self.pond_pollution / 100
        
        sky_b = int(150 + (255 - 150) * (1 - pollution_ratio))
        sky_g = int(150 + (200 - 150) * (1 - pollution_ratio))
        sky_r = int(150 + (150 - 150) * (1 - pollution_ratio))
        
        pond[:, :] = (sky_b, sky_g, sky_r)  # 하늘색 배경
        
        # 연못 오염도에 따른 색상 계산
        # 오염됨 (100) -> 갈색: (50, 100, 100)
        # 깨끗함 (0) -> 청록색: (200, 255, 100)
        pollution_ratio = self.pond_pollution / 100
        
        r = int(50 + (200 - 50) * (1 - pollution_ratio))
        g = int(100 + (255 - 100) * (1 - pollution_ratio))
        b = int(100 + (100 - 100) * (1 - pollution_ratio))
        
        # 연못 타원 그리기
        pond_center = (width // 2, int(height * 0.6))
        pond_axes = (int(width * 0.4), int(height * 0.35))
        cv.ellipse(pond, pond_center, pond_axes, 0, 0, 360, (b, g, r), -1)
        
        # 연못 테두리
        cv.ellipse(pond, pond_center, pond_axes, 0, 0, 360, (80, 80, 80), 3)
        
        # 오염도 표시
        if self.pond_pollution > 60:
            status = "POLLUTED"
            status_color = (0, 0, 255)
            # 오염 표시 (쓰레기)
            for _ in range(int(self.pond_pollution / 10)):
                x = random.randint(pond_center[0] - pond_axes[0] + 50, 
                                 pond_center[0] + pond_axes[0] - 50)
                y = random.randint(pond_center[1] - pond_axes[1] + 20, 
                                 pond_center[1] + pond_axes[1] - 20)
                cv.circle(pond, (x, y), random.randint(3, 8), (0, 0, 0), -1)
        elif self.pond_pollution > 30:
            status = "CLEANING..."
            status_color = (0, 165, 255)
        else:
            status = "CLEAN!"
            status_color = (0, 255, 0)
            # 물고기 추가
            for _ in range(5):
                x = random.randint(pond_center[0] - pond_axes[0] + 50, 
                                 pond_center[0] + pond_axes[0] - 50)
                y = random.randint(pond_center[1] - pond_axes[1] + 20, 
                                 pond_center[1] + pond_axes[1] - 20)
                cv.circle(pond, (x, y), 5, (0, 255, 255), -1)
        
        # 상태 텍스트
        pond = self.put_korean_text(pond, status, (width // 2 - 80, 35), 30, status_color)
        
        # 오염도 바
        bar_x = 50
        bar_y = height - 50
        bar_width = width - 100
        bar_height = 30
        
        # 바 배경
        cv.rectangle(pond, (bar_x, bar_y), (bar_x + bar_width, bar_y + bar_height), 
                    (200, 200, 200), -1)
        
        # 오염도 바 채우기
        fill_width = int(bar_width * (self.pond_pollution / 100))
        fill_color = (0, int(255 * (1 - pollution_ratio)), int(255 * pollution_ratio))
        cv.rectangle(pond, (bar_x, bar_y), (bar_x + fill_width, bar_y + bar_height), 
                    fill_color, -1)
        
        # 오염도 텍스트
        pond = self.put_korean_text(pond, f"오염도: {self.pond_pollution}%", 
                                    (bar_x, bar_y - 15), 18, (0, 0, 0))
        
        # 연못이 깨끗하면 연꽃 추가
        if self.pond_pollution == 0:
            self.draw_lotus(pond, pond_center[0] - 80, pond_center[1] - 30)
        
        return pond
    
    def draw_lotus(self, img, x, y):
        """연꽃 그리기"""
        # 연잎
        cv.ellipse(img, (x, y + 10), (25, 15), 0, 0, 360, (50, 150, 50), -1)
        cv.ellipse(img, (x, y + 10), (25, 15), 0, 0, 360, (30, 100, 30), 2)
        
        # 연꽃 꽃잎 
        petal_color = (180, 120, 255)
        
        # 5개의 꽃잎
        for angle in range(0, 360, 72):
            rad = np.radians(angle)
            petal_x = int(x + 12 * np.cos(rad))
            petal_y = int(y + 12 * np.sin(rad))
            cv.ellipse(img, (petal_x, petal_y), (8, 12), angle, 0, 360, petal_color, -1)
        
        # 꽃 중심 
        cv.circle(img, (x, y), 6, (0, 200, 255), -1)
    
    
    def draw_quiz_screen(self):
        """퀴즈 화면 그리기"""
        if self.current_question >= len(self.selected_questions):
            return self.draw_result_screen()
        
        screen = np.ones((self.screen_height, self.screen_width, 3), dtype=np.uint8) * 240
        
        # 상단 진행도
        progress_text = f"문제 {self.current_question + 1}/{self.total_questions} | 점수: {self.score}"
        screen = self.put_korean_text(screen, progress_text, (20, 25), 20, (0, 0, 0))
        
        # 연못 미리보기
        mini_pond = self.draw_pond(300, 150)
        screen[50:200, 350:650] = mini_pond
        
        # 질문
        question = self.selected_questions[self.current_question]
        
        # 질문 박스
        y_offset = 220
        cv.rectangle(screen, (15, y_offset - 25), (685, y_offset + 60), (255, 255, 255), -1)
        cv.rectangle(screen, (15, y_offset - 25), (685, y_offset + 60), (100, 100, 100), 2)
        
        # 질문 텍스트 (두 줄로 나누기)
        q_text = question["question"]
        if len(q_text) > 30:
            mid = len(q_text) // 2
            space_idx = q_text.find(' ', mid)
            if space_idx != -1:
                line1 = q_text[:space_idx]
                line2 = q_text[space_idx+1:]
                screen = self.put_korean_text(screen, line1, (25, y_offset), 18, (0, 0, 0))
                screen = self.put_korean_text(screen, line2, (25, y_offset + 30), 18, (0, 0, 0))
            else:
                screen = self.put_korean_text(screen, q_text, (25, y_offset + 10), 18, (0, 0, 0))
        else:
            screen = self.put_korean_text(screen, q_text, (25, y_offset + 10), 18, (0, 0, 0))
        
        # 선택지
        y_offset = 310
        for i, option in enumerate(question["options"]):
            y = y_offset + i * 60
            
            # 선택지 박스
            cv.rectangle(screen, (30, y), (670, y + 50), (255, 255, 255), -1)
            cv.rectangle(screen, (30, y), (670, y + 50), (150, 150, 150), 2)
            
            # 번호와 텍스트
            text = f"{i+1}. {option}"
            screen = self.put_korean_text(screen, text, (45, y + 18), 18, (0, 0, 0))
        
        # 안내 텍스트
        screen = self.put_korean_text(screen, "1-4번 키로 답변하세요", (20, 575), 16, (100, 100, 100))
        
        return screen
    
    def check_answer(self, user_answer):
        """답안 확인 및 피드백"""
        question = self.selected_questions[self.current_question]
        correct = (user_answer == question["answer"])
        
        if correct:
            self.score += 1
            # 연못 정화 (맞출 때마다 20%씩 감소)
            self.pond_pollution = max(0, self.pond_pollution - 20)
            feedback = "정답입니다!"
            color = (0, 255, 0)
        else:
            feedback = "틀렸습니다!"
            color = (0, 0, 255)
        
        # 피드백 화면
        screen = np.ones((self.screen_height, self.screen_width, 3), dtype=np.uint8) * 240
        
        # 결과 텍스트
        screen = self.put_korean_text(screen, feedback, (250, 80), 40, color)
        
        # 정답
        answer_text = f"정답: {question['options'][question['answer']]}"
        screen = self.put_korean_text(screen, answer_text, (60, 150), 22, (0, 0, 0))
        
        # 설명 박스
        y_offset = 200
        cv.rectangle(screen, (30, y_offset - 15), (670, y_offset + 70), (255, 255, 255), -1)
        cv.rectangle(screen, (30, y_offset - 15), (670, y_offset + 70), (100, 100, 100), 2)
        
        # 설명 텍스트 (여러 줄)
        explanation = question["explanation"]
        if len(explanation) > 35:
            mid = len(explanation) // 2
            space_idx = explanation.find(' ', mid)
            if space_idx != -1:
                line1 = explanation[:space_idx]
                line2 = explanation[space_idx+1:]
                screen = self.put_korean_text(screen, line1, (45, y_offset + 5), 17, (0, 0, 0))
                screen = self.put_korean_text(screen, line2, (45, y_offset + 35), 17, (0, 0, 0))
            else:
                screen = self.put_korean_text(screen, explanation, (45, y_offset + 20), 17, (0, 0, 0))
        else:
            screen = self.put_korean_text(screen, explanation, (45, y_offset + 20), 17, (0, 0, 0))
        
        # 연못 상태
        pond = self.draw_pond(640, 270)
        screen[310:580, 30:670] = pond
        
        # 다음 안내
        screen = self.put_korean_text(screen, "스페이스바를 눌러 계속하세요", (200, 590), 16, (100, 100, 100))
        
        cv.imshow('Waste Quiz Game', screen)
        
        # 스페이스바 대기
        while True:
            key = cv.waitKey(100) & 0xFF
            if key == ord(' '):
                break
            elif key == ord('q'):
                return False
        
        self.current_question += 1
        return True
    
    def draw_result_screen(self):
        """최종 결과 화면"""
        screen = np.ones((self.screen_height, self.screen_width, 3), dtype=np.uint8) * 240
        
        # 최종 점수
        screen = self.put_korean_text(screen, "퀴즈 완료!", (260, 50), 35, (0, 0, 0))
        
        score_text = f"최종 점수: {self.score}/{self.total_questions}"
        screen = self.put_korean_text(screen, score_text, (220, 100), 28, (0, 0, 0))
        
        # 결과 메시지
        if self.score >= 3:
            message = "연못이 정화되었습니다!"
            message_color = (35, 122, 75)
            self.pond_pollution = 0
        else:
            message = "연못이 아직 오염되어 있습니다"
            message_color = (0, 0, 255)
        
        screen = self.put_korean_text(screen, message, (150, 150), 26, message_color)
        
        # 최종 연못 상태
        pond = self.draw_pond(640, 350)
        screen[200:550, 30:670] = pond
        
        # 재시작 안내
        screen = self.put_korean_text(screen, "R: 재시작 | Q: 종료", (230, 580), 18, (100, 100, 100))
        
        return screen
    
    def run(self):
        """게임 실행"""
        print("🎮 연못 정화 퀴즈 게임 시작!")
        print("=" * 50)
        print("📝 5개 문제 중 3개 이상 맞추면 연못이 정화됩니다!")
        print("🎯 키 조작: 1-4(답변), SPACE(다음), R(재시작), Q(종료)")
        print("=" * 50)
        
        self.select_random_questions()
        
        while True:
            screen = self.draw_quiz_screen()
            cv.imshow('Waste Quiz Game', screen)
            
            # 결과 화면이면 다른 처리
            if self.current_question >= len(self.selected_questions):
                while True:
                    key = cv.waitKey(100) & 0xFF
                    if key == ord('r'):
                        self.select_random_questions()
                        break
                    elif key == ord('q'):
                        cv.destroyAllWindows()
                        return
                continue
            
            # 답변 입력 대기
            key = cv.waitKey(1) & 0xFF
            
            if key == ord('q'):
                break
            elif key in [ord('1'), ord('2'), ord('3'), ord('4')]:
                user_answer = key - ord('1')
                if not self.check_answer(user_answer):
                    break
        
        cv.destroyAllWindows()
        print("👋 게임을 종료합니다.")


def main():
    """메인 실행 함수"""
    game = WasteQuizGame()
    game.run()


if __name__ == "__main__":
    main()

🎮 연못 정화 퀴즈 게임 시작!
📝 5개 문제 중 3개 이상 맞추면 연못이 정화됩니다!
🎯 키 조작: 1-4(답변), SPACE(다음), R(재시작), Q(종료)
